# 1. Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from IPython.display import display, Markdown
import ipywidgets as widgets
%load_ext autoreload
%autoreload 2


from Dataframes import dataframe_train, dafaframe_test


# --------------------
# OPÇÕES DE MODELOS:
# --------------------
from sklearn.linear_model import LassoCV
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import LinearRegression


# 2. Extração de dados (48 Train e 19 Test)

In [ ]:
from Dataframes import dafaframe_test, dataframe_train

In [ ]:
# dataframe_train()
# dafaframe_test()
# transforma_df_em_csv()
# fazer_novo_df()

In [ ]:
print("Carregando bases de dados em CSV...")
df_treino_csv = pd.read_csv('Dados_Processados/treino_features.csv')
df_teste_csv = pd.read_csv('Dados_Processados/teste_features.csv')

# ========================================================
# O  DE SALVAMENTO: Recriando o Porta_ID caso ele não exista!
# Toda vez que o 'Ciclo' cai (ex: vai de 150 de volta para 1), ele soma +1 no ID.
# ========================================================
if 'Porta_ID' not in df_treino_csv.columns:
    print("Recriando a coluna Porta_ID no Treino...")
    df_treino_csv['Porta_ID'] = (df_treino_csv['Ciclo'] < df_treino_csv['Ciclo'].shift(1)).cumsum() + 1

if 'Porta_ID' not in df_teste_csv.columns:
    print("Recriando a coluna Porta_ID no Teste...")
    df_teste_csv['Porta_ID'] = (df_teste_csv['Ciclo'] < df_teste_csv['Ciclo'].shift(1)).cumsum() + 1

# 2. O truque: o groupby converte o DataFrame gigante de volta para a estrutura: [(1, df_porta1), ...]
portas_treino = list(df_treino_csv.groupby('Porta_ID'))
portas_teste = list(df_teste_csv.groupby('Porta_ID'))

print(f"✅ Sucesso! Encontradas {len(portas_treino)} portas de Treino e {len(portas_teste)} portas de Teste.")

# 3. Modelo

In [ ]:
import os

total = 0
for i in range(1, 20):
    nome_pasta = f"Test/Test_{i}" 
    arquivos_csv = [arq for arq in os.listdir(nome_pasta) if arq.endswith('.csv')]        
    print(f"{i}: {len(arquivos_csv)}, ")
    total += len(arquivos_csv)
print(f"TOTAL: {total}")

In [ ]:
X_train_list, y_train_list = [], []

# ====================================================================
# PASSO 1: PREPARAÇÃO DOS DADOS DE TREINO
# ====================================================================
# Lista unificada de colunas para remover do modelo (Evita o erro de Feature Missing!)
colunas_para_ignorar = ['Porta_ID', 'Ciclo', 'Ciclo_Relativo', 'RUL_Gabarito']

for porta_id, df_porta in portas_treino:
    df_features = df_porta.copy() 
    
    # 1. Calcula o RUL real (O alvo do modelo)
    if 'RUL_Gabarito' in df_features.columns:
        target = df_features['RUL_Gabarito']
    else:
        ciclo_da_falha = df_features['Ciclo'].max()
        target = ciclo_da_falha - df_features['Ciclo']
    
    # 2. Removemos a resposta e os relógios. O modelo foca apenas nos sensores!
    features = df_features.drop(columns=colunas_para_ignorar, errors='ignore')
    
    X_train_list.append(features)
    y_train_list.append(target)

# Junta todas as portas numa única matriz de aprendizagem
X_train_full = pd.concat(X_train_list, ignore_index=True).fillna(0)
y_train_full = np.concatenate(y_train_list)

print("A ajustar os hiperparâmetros (Cross-Validation) e treinando...")

# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
from sklearn.ensemble import RandomForestRegressor
modelo_elite = RandomForestRegressor(
    n_estimators=500,        # Mais árvores para estabilizar a previsão
    max_depth=12,            # Altura controlada para generalizar melhor
    min_samples_leaf=3,      # Evita que o modelo decore ruídos
    max_features='sqrt',     # Reduz a dependência de uma só variável
    random_state=42,
    n_jobs=-1
)

print("🚂 Treinando com Foco no Horizonte de Falha...")
modelo_elite.fit(X_train_full, y_train_full)

# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!




# ====================================================================
# PASSO 2: PREPARAÇÃO DOS DADOS DE TESTE
# ====================================================================
X_test_list = []

for porta_id, df_porta in portas_teste:
    df_features_teste = df_porta.copy() 
    
    # Removemos EXATAMENTE as mesmas colunas do treino
    features_teste = df_features_teste.drop(columns=colunas_para_ignorar, errors='ignore')
    
    X_test_list.append(features_teste)

# Junta todos os testes numa única matriz
X_test_full = pd.concat(X_test_list, ignore_index=True).fillna(0)

# ====================================================================
# PASSO 3: A PREDIÇÃO
# ====================================================================
print("Realizando as predições na base de Teste...")
RUL_predito = modelo_elite.predict(X_test_full)
print("✅ Predições concluídas!")

## Submission.csv

In [ ]:
# ====================================================================
# PASSO 2: PREPARAÇÃO DOS DADOS DE TESTE
# ====================================================================
X_test_list = []

for porta_id, df_porta in portas_teste:
    df_features_teste = df_porta.copy() 
    
    # Removemos EXATAMENTE as mesmas colunas do treino
    features_teste = df_features_teste.drop(columns=colunas_para_ignorar, errors='ignore')
    
    X_test_list.append(features_teste)

# Junta todos os testes numa única matriz
X_test_full = pd.concat(X_test_list, ignore_index=True).fillna(0)

# ====================================================================
# PASSO 3: A PREDIÇÃO
# ====================================================================
print("Realizando as predições na base de Teste...")
RUL_predito = modelo_elite.predict(X_test_full)
print("✅ Predições concluídas!")
# ====================================================================
# PASSO 4: MONTANDO OS DADOS COM PREENCHIMENTO E GERANDO SUBMISSION
# ====================================================================
gabaritos_reais = {
    1: 3593, 2: 2358, 3: 506, 4: 2700, 5: 6458, 6: 2265, 7: 1135, 8: 1855, 9: 1778,\
    10: 3384, 11: 3532, 12: 2469, 13: 2177, 14: 2119, 15: 2526, 16: 1709, 17: 2323, 18: 2784, 19: 1516
}

# 1. Tratamos as predições do modelo PRIMEIRO (Evita que o RUL seja negativo ou decimal)
# Assim não corremos o risco de transformar os "0" do preenchimento em "1" depois.
RUL_tratado = np.clip(RUL_predito, 1, None).round().astype(int)

ids_teste = []
rul_alinhado = []
indice_corte = 0

# 2. Inserindo os zeros diretamente na formação das listas
for porta_id, df in portas_teste:
    tamanho_atual = len(df)
    linhas_faltantes = gabaritos_reais[porta_id] - tamanho_atual
    
    # Recorta exatamente as predições correspondentes a esta porta
    preds_reais_porta = RUL_tratado[indice_corte : indice_corte + tamanho_atual]
    
    # Se faltam linhas, adicionamos os IDs e os Zeros ANTES
    if linhas_faltantes > 0:
        ids_teste.extend([porta_id] * linhas_faltantes)
        rul_alinhado.extend([0] * linhas_faltantes)
        
    # Logo abaixo, adicionamos os IDs e os valores REAIS que o modelo previu
    ids_teste.extend([porta_id] * tamanho_atual)
    rul_alinhado.extend(preds_reais_porta)
    
    # Atualiza o ponto de corte para a próxima porta
    indice_corte += tamanho_atual

# 3. Sobrescrevemos a variável original conforme o seu pedido
RUL_predito = rul_alinhado

# 4. Criamos o df_resultados APENAS com a 1ª e 3ª colunas (ID e RUL)
df_resultados = pd.DataFrame({
    'Porta_ID': ids_teste,
    'RUL_Final': RUL_predito
})

print("✅ Preenchimento de zeros realizado direto no RUL_predito!")
print(f"Total de ciclos originais alinhados: {len(df_resultados)}")


#### Criação do submission:

In [ ]:
# A quantidade de linhas finais exigidas por porta no Data Challenge
base = {
    1: 3593, 2: 2358, 3: 506, 4: 2700, 5: 6458, 6: 2265, 7: 1135, 8: 1855, 9: 1778,
    10: 3384, 11: 3532, 12: 2469, 13: 2177, 14: 2119, 15: 2526, 16: 1709, 17: 2323, 18: 2784, 19: 1516
}

RUL_final = []
ids_finais = []
indice_corte = 0

tamanho = 0

for porta_id, df in portas_teste:
    # 1. Identifica o tamanho original que a porta tem no seu RUL_predito atual
    tamanho_original = len(df)


    tamanho += base[porta_id]
    print(f"{porta_id}: {base[porta_id]}")

    
    # 2. Recorta apenas as predições correspondentes a essa porta
    preds_porta = RUL_predito[indice_corte : indice_corte + tamanho_original]
    
    # REGRA 1: Retirar o último valor (que é 0, fazendo terminar em 1)
    # Transforma em lista e fatia excluindo o último elemento [:-1]
    preds_porta_sem_ultimo = list(preds_porta)[:-1]
    
    # REGRA 2: Dobrar cada linha (ex: 4, 3, 2, 1 -> 4, 4, 3, 3, 2, 2, 1, 1)
    preds_dobradas = []
    for valor in preds_porta_sem_ultimo:
        preds_dobradas.extend([valor, valor])
        
    # REGRA 3: Preencher com "0" antes do primeiro valor até atingir o tamanho 'base'
    tamanho_atual_dobrado = len(preds_dobradas)
    linhas_faltantes = base[porta_id] - tamanho_atual_dobrado
    
    if linhas_faltantes > 0:
        # Cria a lista de zeros e soma (concatena) com a lista de predições dobradas
        preds_finais_porta = ([0] * linhas_faltantes) + preds_dobradas
    else:
        # Caso já tenha atingido o tamanho (proteção de segurança)
        preds_finais_porta = preds_dobradas
        
    # 4. Adiciona o resultado da porta nas listas mestres do submission
    RUL_final.extend(preds_finais_porta)
    ids_finais.extend([porta_id] * len(preds_finais_porta))
    
    # Atualiza o índice para cortar corretamente a predição da próxima porta
    indice_corte += tamanho_original

print(f"TOTAL: {tamanho}")


# ====================================================================
# GERANDO O ARQUIVO DE SUBMISSÃO FINAL
# ====================================================================
# Monta a tabela perfeitamente alinhada
df_submission = pd.DataFrame({
    'Porta_ID': ids_finais,
    'RUL_Final': RUL_final
})

# Salva no formato do Data Challenge
nome_arquivo = 'submission.csv'
df_submission.to_csv(nome_arquivo, sep=';', header=False, index=False)

print(f"✅ 'RUL_final' criado e '{nome_arquivo}' gerado com sucesso!")
print(f"TAMANHO SUBMISSION: {len(df_submission)}")

# SCORE

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import os
from IPython.display import display, HTML

def norm_f(m, a=4443.76, b=1.53, c=4443.76, d=0):
    return (a / ((m**b) + c)) + d



df_gab = pd.read_csv("gabarito_oficial.csv", sep=';', header=None)
df_sub = pd.read_csv("submission.csv", sep=';', header=None)

df_score = pd.DataFrame({
    'Porta_ID': df_gab[0],
    'RUL_Real': df_gab[1],
    'RUL_Pred': df_sub[1]
})


resultados_gerais = []
alpha_peso = 2.0

for porta in df_score['Porta_ID'].unique():
    print(f"\n{'-'*40}")
    print(f"🚪 AVALIANDO PORTA: {porta}")
    print(f"{'-'*40}")
    
    # 1. Filtramos apenas os dados desta porta e resetamos o index
    # Resetar o index garante que o t_alpha.index[0] comece do 0 para cada porta
    df_porta = df_score[df_score['Porta_ID'] == porta].reset_index(drop=True)
    
    # ========================================================
    # SUAS EQUAÇÕES MATEMÁTICAS EXATAS
    # ========================================================
    df_porta['Erro'] = (df_porta['RUL_Real'] - df_porta['RUL_Pred']).abs()
    
    N = len(df_porta) # Ajustado para ler o tamanho da porta, não do df inteiro
        
    # Usando a versão com np.sum para o RMSE virar um escalar (número único)
    rmse_bruto = np.sqrt(np.sum(df_porta['Erro']**2)/N)

    erro_relativo = df_porta['Erro'] / df_porta['RUL_Real']
    x = (erro_relativo <= 0.1).sum()
    precision_bruto = 100*(x/N)

    condicao_atingida = erro_relativo <= 0.2

    if condicao_atingida.any():
        # Pega a linha exata em que atingiu a margem para esta porta
        t_alpha = df_porta[condicao_atingida].index[0]
        print(f"T_alpha: {t_alpha} (de {N} ciclos totais)")
    else:
        # Se nunca atingir, t_alpha vira N para o PH ser 0 (se fosse None, a equação abaixo daria erro)
        t_alpha = N 
        print("❌ O modelo NUNCA atingiu a margem de 20%.")
    
    ph_bruto = (N - t_alpha)/N

    # Normalização
    rmse_norm = norm_f(rmse_bruto)
    precision_norm = precision_bruto

    # Score
    score = (rmse_norm + precision_norm + alpha_peso*ph_bruto) / (2 + alpha_peso)

    # ========================================================
    # PRINT DOS RESULTADOS INDIVIDUAIS
    # ========================================================
    print(f"Score Final : {score:.4f}")
    print(f"RMSE (Norm) : {rmse_norm:.4f}")
    print(f"Precision   : {precision_norm:.4f}")
    print(f"PH          : {ph_bruto:.4f}")
    
    # Salvando no dicionário para a média global
    resultados_gerais.append({
        'Porta_ID': porta,
        'Score': score,
        'RMSE': rmse_norm,
        'Precision': precision_norm,
        'PH': ph_bruto
    })

# ========================================================
# CÁLCULO DA MÉDIA GERAL (DAS 19 PORTAS)
# ========================================================
# Transformamos a lista de dicionários numa tabela para facilitar o cálculo da média
df_resultados = pd.DataFrame(resultados_gerais)

media_score = df_resultados['Score'].mean()
media_rmse = df_resultados['RMSE'].mean()
media_precision = df_resultados['Precision'].mean()
media_ph = df_resultados['PH'].mean()

print("\n" + "="*50)
print("🏆 MÉDIA GLOBAL (TODAS AS PORTAS) 🏆")
print("="*50)
print(f"Score Médio Geral : {media_score:.4f}")
print(f"RMSE Médio        : {media_rmse:.4f}")
print(f"Precision Médio   : {media_precision:.4f}")
print(f"PH Médio          : {media_ph:.4f}")
print("="*50)

# Gráfico

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# 1. Pega todas as portas únicas
portas = df['Porta_ID'].unique()

# 2. Define o layout do painel (5 linhas x 4 colunas = 20 espaços, cabem as 19 portas)
linhas = 5
colunas = 4

# 3. Cria a estrutura dos subplots (o grid)
fig = make_subplots(
    rows=linhas, cols=colunas, 
    subplot_titles=[f"<b>Porta {p}</b>" for p in portas],
    vertical_spacing=0.06,   
    horizontal_spacing=0.04  
)

# 4. Loop para preencher cada "quadradinho" do grid
for i, porta in enumerate(portas):
    # Isola os dados da porta e recria o index
    df_porta = df[df['Porta_ID'] == porta].reset_index(drop=True)
    
    # CORREÇÃO AQUI: Transformando o range em uma lista
    ciclos = list(range(1, len(df_porta) + 1))
    
    # Calcula em qual linha e coluna este gráfico deve entrar
    linha_atual = (i // colunas) + 1
    coluna_atual = (i % colunas) + 1
    
    # Truque: Mostra a legenda apenas no primeiro gráfico para não poluir a tela
    mostrar_legenda = True if i == 0 else False
    
    # LINHA 1: RUL REAL (Gabarito) - Verde Tracejado
    fig.add_trace(
        go.Scatter(x=ciclos, y=df_porta['RUL_Real'], mode='lines', 
                   name='RUL Real', line=dict(color='#2ca02c', width=2, dash='dash'), 
                   showlegend=mostrar_legenda),
        row=linha_atual, col=coluna_atual
    )
    
    # LINHA 2: RUL PREDITO (Modelo) - Laranja Sólido
    fig.add_trace(
        go.Scatter(x=ciclos, y=df_porta['RUL_Pred'], mode='lines', 
                   name='RUL Predito', line=dict(color='#ff7f0e', width=2), 
                   showlegend=mostrar_legenda),
        row=linha_atual, col=coluna_atual
    )

# 5. Embelezamento do Layout Geral
fig.update_layout(
    title=dict(
        text='<b>Painel Completo: RUL Real vs Predito (Todas as 19 Portas)</b>', 
        x=0.5, font=dict(size=22, family="Bookman Old Style, serif")
    ),
    height=1200, 
    plot_bgcolor='white',
    font=dict(family="Bookman Old Style, serif"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5)
)

fig.update_xaxes(showgrid=True, gridcolor='#E5E5E5', zeroline=False)
fig.update_yaxes(showgrid=True, gridcolor='#E5E5E5', zeroline=False)

fig.show()